# Загрузка курсов ЦБ в Postgres

**Что делает ноутбук:** скачивает `daily_json.js` с cbr-xml-daily.ru и пишет курсы в `cbr.valute_course`.

**Требования, которые он закрывает:**

1. **Идемпотентность** — `ON CONFLICT (date_req, char_code) DO NOTHING`. Повторный запуск за ту же дату не создаёт дублей; проверка в конце показывает `Вставлено: 0`.
2. **Обработка ошибок** — каждый `try/except` вокруг сети, `.env` и БД, с понятным русским сообщением и `from e`, чтобы не терять исходный трейс.
3. **Параметры через `.env`** — ни логина, ни пароля, ни хоста в коде.
4. **Читаемость** — осмысленные имена, комментарии к неочевидным местам (`Decimal`, `nominal`, `ON CONFLICT`).

**Предусловие:** в таблице `cbr.valute_course` должен быть уникальный индекс `(date_req, char_code)`. Если его нет — ячейка «Гарантируем уникальный индекс» создаст его идемпотентно.

In [24]:
import os
from decimal import Decimal
from pathlib import Path

import requests
import psycopg
from dotenv import load_dotenv

## 1. Читаем `.env`

Файл должен лежать **рядом с ноутбуком**. Если его нет — падаем сразу и с понятным сообщением, а не `KeyError` через три ячейки.

In [25]:
env_path = Path(".env")
if not env_path.exists():
    raise FileNotFoundError(
        f"Не найден {env_path.resolve()}. "
        f"Скопируйте .env.example в .env и задайте пароль."
    )

load_dotenv(env_path)

True

## 2. Параметры подключения

Проверяем, что обязательные переменные заданы, и только потом собираем `dict` для `psycopg.connect`. `POSTGRES_HOST` по умолчанию `127.0.0.1` — ноутбук запускается **на хосте**, а не в compose-сети, и стучится в опубликованный `127.0.0.1:5432`.

In [26]:
REQUIRED_ENV = ("POSTGRES_USER", "POSTGRES_PASSWORD", "POSTGRES_DB")
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f"В .env не заданы: {', '.join(missing)}")

DB = dict(
    host=os.getenv("POSTGRES_HOST", "127.0.0.1"),
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname=os.environ["POSTGRES_DB"],
    connect_timeout=5,  # не ждём 30+ секунд, если хост недоступен
)

print(f"DB: {DB['user']}@{DB['host']}:{DB['port']}/{DB['dbname']}")

DB: cbr@127.0.0.1:5432/cbr


## 3. Скачиваем данные

Таймаут обязателен: без него `requests.get` может висеть бесконечно, и ноутбук «зависнет». `raise_for_status()` превращает 4xx/5xx в исключение, а не молча пишет мусор.

In [27]:
URL = "https://www.cbr-xml-daily.ru/daily_json.js"

try:
    response = requests.get(URL, timeout=15)
    response.raise_for_status()
    payload = response.json()
except requests.RequestException as exc:
    raise RuntimeError(f"Не удалось получить данные ЦБ: {exc}") from exc

date_req = payload["Date"]          # "YYYY-MM-DD"
valutes = payload["Valute"]         # dict: ключ = CharCode, значение = валюта

print(f"Дата курса: {date_req}, валют: {len(valutes)}")

Дата курса: 2026-11-14T11:30:00+03:00, валют: 54


## 4. Преобразуем в строки для вставки

- `Decimal(str(...))` — деньги. `float` теряет копейки на округлении, `NUMERIC` в БД ждёт `Decimal`.
- `nominal` — за сколько единиц валюты дан курс (USD → 1, JPY → 100). `value / nominal` = рублей за единицу. Без этого курсы сравнивать нельзя.
- `previous` — может отсутствовать у новых валют, поэтому `None` разрешён.

In [28]:
rows = [
    (
        v["ID"],
        v["NumCode"],
        v["CharCode"],
        v["Nominal"],
        v["Name"],
        Decimal(str(v["Value"])),
        Decimal(str(v["Previous"])) if v.get("Previous") is not None else None,
        date_req,
    )
    for v in valutes.values()
]

In [29]:
print(f"Подготовлено строк: {len(rows)}")
print("Пример:", rows[0])

Подготовлено строк: 54
Пример: ('R01010', '036', 'AUD', 1, 'Австралийский доллар', Decimal('55.8959'), Decimal('56.7668'), '2026-11-14T11:30:00+03:00')


## 5. Гарантируем уникальный индекс

`ON CONFLICT (date_req, char_code)` не работает без **уникального** индекса по этим колонкам — Postgres упадёт с `there is no unique or exclusion constraint matching the ON CONFLICT specification`. Блок идемпотентен: если индекс уже есть — ничего не делает.

In [30]:
ENSURE_UNIQUE_SQL = """
DO $$
BEGIN
    IF NOT EXISTS (
        SELECT 1
        FROM pg_indexes
        WHERE schemaname = 'cbr'
          AND tablename  = 'valute_course'
          AND indexname  = 'valute_course_date_char_uq'
    ) THEN
        ALTER TABLE cbr.valute_course
            ADD CONSTRAINT valute_course_date_char_uq
            UNIQUE (date_req, char_code);
    END IF;
END $$;
"""

try:
    with psycopg.connect(**DB) as conn, conn.cursor() as cur:
        cur.execute(ENSURE_UNIQUE_SQL)
except psycopg.Error as exc:
    raise RuntimeError(
        f"Не удалось создать уникальный индекс (date_req, char_code): {exc}"
    ) from exc

print("Уникальный индекс (date_req, char_code) на месте")

Уникальный индекс (date_req, char_code) на месте


## 6. Вставляем идемпотентно

`ON CONFLICT ... DO NOTHING` — ключевое: повторная вставка той же `(date_req, char_code)` просто пропускается. `cur.rowcount` после `executemany` показывает, сколько строк **реально** добавлено (пропущенные не считаются). `with psycopg.connect(...)` коммитит при выходе и откатывает при исключении — ручной `commit()` не нужен.

In [31]:
INSERT_SQL = """
INSERT INTO cbr.valute_course
    (valute_id, num_code, char_code, nominal, name, value, previous, date_req)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (date_req, char_code) DO NOTHING
"""

try:
    with psycopg.connect(**DB) as conn, conn.cursor() as cur:
        cur.executemany(INSERT_SQL, rows)
        inserted = cur.rowcount

        cur.execute(
            "SELECT COUNT(*) FROM cbr.valute_course WHERE date_req = %s",
            (date_req,),
        )
        total_for_date = cur.fetchone()[0]
except psycopg.Error as exc:
    raise RuntimeError(f"Ошибка записи в БД: {exc}") from exc

print(f"Вставлено: {inserted}, всего за {date_req}: {total_for_date}")

Вставлено: 0, всего за 2026-11-14T11:30:00+03:00: 54


## 7. Проверка

- **Первый запуск:** `Вставлено` = числу валют (~45).
- **Повторный запуск тех же ячеек:** `Вставлено: 0`, `всего` не меняется — это и есть идемпотентность.

Ниже — выборка «рубли за единицу валюты» (`value / nominal`), чтобы не сравнивать JPY за 100 с USD за 1.

In [32]:
try:
    with psycopg.connect(**DB) as conn, conn.cursor() as cur:
        cur.execute(
            """
            SELECT char_code, nominal, value, value / nominal AS rub_per_unit
            FROM cbr.valute_course
            WHERE date_req = %s
            ORDER BY char_code
            LIMIT 10
            """,
            (date_req,),
        )
        for char_code, nominal, value, rub_per_unit in cur.fetchall():
            print(f"{char_code:4} nominal={nominal:>4}  value={value:>12}  rub/unit={rub_per_unit:.6f}")
except psycopg.Error as exc:
    raise RuntimeError(f"Ошибка чтения из БД: {exc}") from exc

AED  nominal=   1  value=   22.369400  rub/unit=22.369400
AMD  nominal= 100  value=   22.785700  rub/unit=0.227857
AUD  nominal=   1  value=   55.895900  rub/unit=55.895900
AZN  nominal=   1  value=   48.324500  rub/unit=48.324500
BDT  nominal= 100  value=   65.794200  rub/unit=0.657942
BHD  nominal=   1  value=  218.441100  rub/unit=218.441100
BOB  nominal=  10  value=   70.395500  rub/unit=7.039550
BRL  nominal=   1  value=   14.633300  rub/unit=14.633300
BYN  nominal=   1  value=   27.381100  rub/unit=27.381100
CAD  nominal=   1  value=   57.630000  rub/unit=57.630000
